In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import wandb

# 1. Authenticate
wandb.login()

# 2. Download TruthfulQA (Generation task)
print("Downloading TruthfulQA dataset...")
try:
    # This dataset is very stable and almost never goes down
    dataset = load_dataset("truthful_qa", "generation", split="train")
    print("Success! TruthfulQA loaded.")
except Exception as e:
    print(f"Error: {e}")

# 3. Convert to Pandas
df = dataset.to_pandas()

# 4. Data Shaping
# TruthfulQA has 'best_answer' (True) and 'incorrect_answers' (Hallucinations)
# We will take the best answer and one random incorrect answer per row
factual_df = pd.DataFrame({
    'question': df['question'],
    'response': df['best_answer'],
    'is_hallucination': 0
})

# Incorrect answers is a list, so we just take the first one for our binary task
hallucinated_df = pd.DataFrame({
    'question': df['question'],
    'response': df['incorrect_answers'].apply(lambda x: x[0] if len(x) > 0 else ""),
    'is_hallucination': 1
})

final_df = pd.concat([factual_df, hallucinated_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# 5. EDA: Analysis
final_df['response_length'] = final_df['response'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(data=final_df, x='response_length', hue='is_hallucination', bins=30, kde=True, palette="viridis")
plt.title("Phase 1 EDA: Truthful vs. Hallucinated (Word Count)")
plt.show()

print(f"Total samples for Baseline: {len(final_df)}")
display(final_df.head())

Error loading dataset: Dataset 'pku-marshal/HaluEval' doesn't exist on the Hub or cannot be accessed.


NameError: name 'dataset' is not defined